# Введение в PyTorch

**Автор:** ИИ-агент, преподаватель по машинному обучению  
**Уровень:** начинающий–средний (нужно знать основы Python и NumPy)  

После прохождения этого ноутбука вы:
- поймёте, что такое тензоры и как с ними работать в PyTorch;
- научитесь использовать автоградиент (autograd) для автоматического вычисления производных;
- соберёте и обучите свою первую нейросеть на PyTorch;
- увидите, как меняется результат при изменении параметров обучения.

---

## План урока

1. **Подготовка окружения** — установим и импортируем PyTorch, проверим версии.
2. **Базовые примеры: тензоры** — создание, операции, перемещение на GPU.
3. **Автоградиент (autograd)** — как PyTorch «запоминает» операции и считает градиенты.
4. **Первый нейросетевой слой** — `nn.Linear` руками и через `nn.Module`.
5. **Обучение простой сети** — классификация на игрушечных данных.
6. **Чуть сложнее: расширяем пример** — добавляем скрытый слой, меняем активацию.
7. **Практические задания / эксперименты** — 5 задач для самостоятельной работы.
8. **Идеи для дальнейшего изучения** — куда двигаться дальше.

---
## Шаг 1. Установка и подготовка окружения

PyTorch — это библиотека для глубокого обучения от Meta (бывш. Facebook). В Google Colab она уже предустановлена, но мы всё равно проверим версию — это хорошая привычка, потому что API может отличаться между релизами.

In [ ]:
# Устанавливаем PyTorch (в Colab обычно уже есть, но на всякий случай)
# Раскомментируйте следующую строку, если PyTorch не установлен:
# !pip install torch torchvision

Зачем нужен `torchvision`? Это сопутствующая библиотека с готовыми датасетами, моделями и утилитами для компьютерного зрения. Мы будем использовать её для удобства, хотя в этом ноутбуке основная работа — чистый PyTorch.

In [ ]:
import sys
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')

# Проверяем, доступен ли GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')

if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Почему важно проверять версии? Некоторые функции появляются только в определённых версиях. Например, `torch.compile()` появился в PyTorch 2.0. Если вы видите ошибку `AttributeError: module 'torch' has no attribute 'compile'`, скорее всего, у вас старая версия.

Минимальные требования для этого ноутбука: **PyTorch >= 1.9**, **Python >= 3.7**.

---
## Шаг 2. Базовые примеры: тензоры

Тензор — это просто многомерный массив. Если вы работали с NumPy, то уже знакомы с концепцией. Главное отличие тензоров PyTorch: они умеют автоматически вычислять градиенты и могут жить на GPU.

Аналогия: скаляр — это одно число, вектор — строка чисел, матрица — таблица, а тензор — «таблица таблиц». Степень вложенности называют размерностью (rank).

In [ ]:
# Создание тензоров — разные способы

# Из списка Python
a = torch.tensor([1, 2, 3, 4])
print(f'Из списка: {a}')
print(f'Тип: {a.dtype}, размерность: {a.shape}')

# Из NumPy-массива
import numpy as np
np_array = np.array([10, 20, 30])
b = torch.from_numpy(np_array)
print(f'Из NumPy: {b}')

# Специальные тензоры
zeros = torch.zeros(3, 4)   # матрица 3x4 из нулей
ones = torch.ones(2, 3)     # матрица 2x3 из единиц
rand = torch.rand(2, 2)     # случайные числа [0, 1)

print(f'Нули: \n{zeros}')
print(f'Единицы: \n{ones}')
print(f'Случайные: \n{rand}')

Обратите внимание: `torch.tensor()` всегда копирует данные. Если вы хотите создать тензор из NumPy без копирования (они будут делить память), используйте `torch.from_numpy()`. Это быстрее, но изменения в одном отразятся на другом.

In [ ]:
# Операции с тензорами

x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

# Арифметика — поэлементная
print(f'Сложение: {x + y}')
print(f'Умножение на число: {x * 10}')
print(f'Поэлементное умножение: {x * y}')

# Скалярное произведение
print(f'Скалярное произведение: {x.dot(y)}')

# Ресайп (изменение формы) — попробуйте изменить shape!
matrix = torch.arange(12).reshape(3, 4)  # попробуйте изменить на (2, 6) или (4, 3)
print(f'Матрица:\n{matrix}')
print(f'Транспонированная:\n{matrix.T}')

**Что можно менять:**
- Попробуйте `.reshape(2, 6)` или `.reshape(4, 3)` — главное, чтобы общее число элементов совпадало (12).
- Поменяйте `dtype` при создании: `torch.tensor([1, 2], dtype=torch.float32)` vs `torch.int64`.

In [ ]:
# Перемещение тензора на GPU и обратно

x_cpu = torch.randn(1000, 1000)  # случайная матрица на CPU
print(f'Устройство тензора: {x_cpu.device}')

if device == 'cuda':
    x_gpu = x_cpu.to('cuda')
    print(f'Тензор на GPU: {x_gpu.device}')
    
    # На GPU матричное умножение гораздо быстрее!
    import time
    
    # CPU
    start = time.time()
    _ = x_cpu @ x_cpu.T
    cpu_time = time.time() - start
    
    # GPU
    torch.cuda.synchronize()  # ждём завершения предыдущих операций
    start = time.time()
    _ = x_gpu @ x_gpu.T
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    
    print(f'CPU: {cpu_time:.4f}с | GPU: {gpu_time:.4f}с | Ускорение: {cpu_time/gpu_time:.1f}x')
else:
    print('GPU недоступен — продолжаем на CPU. Это нормально для обучения!')

---
## Шаг 3. Автоградиент (autograd)

Это главная «суперсила» PyTorch. Когда вы создаёте тензор с `requires_grad=True`, PyTorch запоминает все операции, которые вы с ним делаете, и может автоматически вычислить градиент (производную) результата по этому тензору.

Зачем это нужно? Обучение нейросети — это по сути оптимизация: мы хотим найти такие веса, при которых ошибка минимальна. Градиент показывает, в какую сторону нужно «подвинуть» веса, чтобы ошибка уменьшилась. Без autograd нам пришлось бы писать производные вручную — это утомительно и чревато ошибками.

Аналогия: представьте, что вы спускаетесь с горы с завязанными глазами. Градиент — это уклон земли под ногами: он говорит, куда шагнуть, чтобы спуститься ниже.

In [ ]:
# Простой пример autograd

# Создаём тензор, для которого будем считать градиент
x = torch.tensor(3.0, requires_grad=True)
print(f'x = {x}')

# Строим функцию: y = x^2 + 2*x + 1
y = x ** 2 + 2 * x + 1
print(f'y = x^2 + 2x + 1 = {y}')

# Вычисляем градиент dy/dx
y.backward()  # запускает «обратный проход» по графу вычислений

# dy/dx = 2x + 2; при x=3 это 8
print(f'dy/dx = {x.grad}')
print(f'Проверка: 2*3 + 2 = {2*3 + 2}')

Как это работает под капотом? PyTorch строит **граф вычислений** — ориентированный ациклический граф, где узлы — операции, а рёбра — тензоры. При вызове `.backward()` PyTorch проходит граф в обратном направлении (от выхода к входу) и применяет правило цепочки производных (chain rule). Это называется **обратным распространением ошибки** (backpropagation).

In [ ]:
# Важный нюанс: градиенты накапливаются!

x = torch.tensor(3.0, requires_grad=True)

for i in range(3):
    y = x ** 2
    y.backward()
    print(f'Шаг {i+1}: градиент = {x.grad}')

# Градиент складывается! Чтобы этого избежать, нужно обнулять:
x.grad.zero_()  # сброс градиента
y = x ** 2
y.backward()
print(f'После сброса: градиент = {x.grad}')

**Запомните:** в цикле обучения градиенты нужно обнулять перед каждым шагом. Иначе вместо текущего градиента вы получите сумму градиентов всех предыдущих шагов. В реальном коде это обычно делает `optimizer.zero_grad()`.

In [ ]:
# Пример с векторным тензором (мини-батч)

W = torch.randn(3, 2, requires_grad=True)  # матрица весов 3x2
b = torch.randn(2, requires_grad=True)      # вектор смещения
x = torch.randn(1, 3)                        # один входной вектор

# Линейное преобразование: y = x @ W + b
y = x @ W + b
loss = y.sum()  # простая функция потерь (для демо)

loss.backward()

print(f'Градиент W:\n{W.grad}')
print(f'Градиент b: {b.grad}')

# Попробуйте изменить размерности: W = torch.randn(5, 3), b = torch.randn(3), x = torch.randn(1, 5)

**Что можно менять:**
- Размерности: `W = torch.randn(5, 3)`, `b = torch.randn(3)`, `x = torch.randn(1, 5)` — главное, чтобы `x.shape[1] == W.shape[0]` и `W.shape[1] == b.shape[0]`.
- Функцию потерь: вместо `.sum()` попробуйте `.mean()` или `.pow(2).mean()`.

---
## Шаг 4. Первый нейросетевой слой: nn.Linear и nn.Module

Мы уже видели линейное преобразование `y = x @ W + b` вручную. PyTorch предоставляет готовый класс `nn.Linear`, который делает то же самое, но удобнее: сам создаёт веса, сам считает градиенты.

А `nn.Module` — это базовый класс для создания любых нейросетей. Он умеет:
- хранить параметры (веса, смещения);
- автоматически собирать все параметры для оптимизатора;
- переключаться между режимами обучения и оценки.

In [ ]:
import torch.nn as nn

# Один линейный слой: 4 входа -> 2 выхода
linear = nn.Linear(in_features=4, out_features=2)  # попробуйте изменить размеры!

# Посмотрим на параметры
print('Параметры слоя:')
for name, param in linear.named_parameters():
    print(f'  {name}: shape={param.shape}')

# Прямой проход
x = torch.randn(1, 4)  # один пример с 4 признаками
y = linear(x)
print(f'\nВход: {x.shape} -> Выход: {y.shape}')
print(f'Результат: {y}')

Теперь создадим полноценную нейросеть с помощью `nn.Module`. Это стандартный способ описания моделей в PyTorch.

In [ ]:
class SimpleNet(nn.Module):
    """Простейшая нейросеть: один скрытый слой + выходной."""
    
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # Определяем слои
        self.fc1 = nn.Linear(input_size, hidden_size)   # вход -> скрытый
        self.relu = nn.ReLU()                           # активация
        self.fc2 = nn.Linear(hidden_size, output_size)  # скрытый -> выход
    
    def forward(self, x):
        """Прямой проход: как данные проходят через сеть."""
        x = self.fc1(x)     # линейное преобразование
        x = self.relu(x)    # нелинейность (без неё сеть = одно линейное преобразование)
        x = self.fc2(x)     # выход
        return x


# Создаём модель
# попробуйте изменить hidden_size, например на 64 или 16
model = SimpleNet(input_size=4, hidden_size=32, output_size=3)
print(model)
print(f'\nВсего параметров: {sum(p.numel() for p in model.parameters()):,}')

Обратите внимание: ReLU (Rectified Linear Unit) — это функция `f(x) = max(0, x)`. Она просто обнуляет отрицательные значения. Зачем она нужна? Без нелинейной активации несколько линейных слоёв эквивалентны одному: `y = W2(W1x + b1) + b2 = (W2W1)x + (W2b1 + b2)`. ReLU «ломает» эту линейность и позволяет сети аппроксимировать сложные функции.

**Что можно менять:**
- `hidden_size` — попробуйте 8, 16, 64, 128. Чем больше — тем больше параметров и выше способность модели, но и выше риск переобучения.
- Активацию: замените `nn.ReLU()` на `nn.Tanh()` или `nn.Sigmoid()` и посмотрите, как изменится обучение.

---
## Шаг 5. Обучение простой сети

Теперь самое интересное — обучим сеть на реальной (хоть и игрушечной) задаче. Мы сгенерируем данные, создадим модель, и «научим» её классифицировать точки.

План обучения:
1. Генерируем данные (спираль из 3 классов — классический бенчмарк).
2. Отдаём данные модели (прямой проход).
3. Считаем ошибку (loss).
4. Считаем градиенты (backward).
5. Обновляем веса (optimizer step).
6. Повторяем много раз.

In [ ]:
# Генерация спирального датасета из 3 классов

import math

def generate_spiral(n_points_per_class=100, n_classes=3):
    """Генерирует спиральный датасет для классификации."""
    X = []
    y = []
    for cls in range(n_classes):
        # Радиус растёт, угол поворачивается — получается спираль
        r = torch.linspace(0, 1, n_points_per_class)
        t = cls * 2 * math.pi / n_classes + r * 4 * math.pi + torch.randn(n_points_per_class) * 0.2
        X.append(torch.stack([r * torch.cos(t), r * torch.sin(t)], dim=1))
        y.append(torch.full((n_points_per_class,), cls, dtype=torch.long))
    X = torch.cat(X, dim=0)
    y = torch.cat(y, dim=0)
    return X, y

torch.manual_seed(42)  # для воспроизводимости
X, y = generate_spiral(n_points_per_class=200, n_classes=3)  # попробуйте изменить количество точек или классов!

print(f'X: {X.shape}, y: {y.shape}')
print(f'Классы: {y.unique().tolist()}')

# Визуализация
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
for cls in range(3):
    mask = y == cls
    plt.scatter(X[mask, 0], X[mask, 1], s=10, label=f'Класс {cls}')
plt.legend()
plt.title('Спиральный датасет')
plt.axis('equal')
plt.show()

In [ ]:
# Полный цикл обучения

torch.manual_seed(42)

# Модель
model = SimpleNet(input_size=2, hidden_size=64, output_size=3)  # 2 признака, 3 класса

# Функция потерь: кросс-энтропия (стандарт для классификации)
criterion = nn.CrossEntropyLoss()

# Оптимизатор: Adam (адаптивный, хорошо работает «из коробки»)
# попробуйте изменить learning_rate: 0.1, 0.01, 0.001, 0.0001
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Количество эпох — сколько раз проходим по всем данным
# попробуйте изменить: 100, 500, 2000
n_epochs = 500

# Для отслеживания прогресса
losses = []
accuracies = []

for epoch in range(n_epochs):
    # 1. Прямой проход
    logits = model(X)  # модель выдаёт «сырые» логиты
    
    # 2. Считаем loss
    loss = criterion(logits, y)
    
    # 3. Обнуляем градиенты (важно!)
    optimizer.zero_grad()
    
    # 4. Обратный проход — считаем градиенты
    loss.backward()
    
    # 5. Обновляем веса
    optimizer.step()
    
    # Логирование
    losses.append(loss.item())
    preds = logits.argmax(dim=1)
    acc = (preds == y).float().mean().item()
    accuracies.append(acc)
    
    if (epoch + 1) % 100 == 0:
        print(f'Эпоха {epoch+1:4d} | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}')

print(f'\nИтого: Loss={losses[-1]:.4f}, Accuracy={accuracies[-1]:.4f}')

Что здесь происходит? Давайте разберём каждый шаг:

1. **Прямой проход** (`model(X)`) — данные проходят через сеть, на выходе получаем логиты (несколько чисел для каждого примера). Логиты — это «уверенность» модели в каждом классе до нормализации.

2. **Loss** (`criterion(logits, y)`) — кросс-энтропия сравнивает логиты с правильными ответами. Чем меньше loss — тем лучше модель угадывает. Кросс-энтропия сначала применяет softmax к логитам (превращает в вероятности), а потом считает `-log(вероятность правильного класса)`.

3. **zero_grad** — обнуляем старые градиенты, иначе они накопятся.

4. **backward** — PyTorch автоматически вычисляет градиенты loss по всем параметрам модели.

5. **step** — оптимизатор обновляет веса по формуле (для Adam это адаптивная версия градиентного спуска).

In [ ]:
# Визуализация процесса обучения

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses)
axes[0].set_title('Loss по эпохам')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies)
axes[1].set_title('Accuracy по эпохам')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация границ решений — как модель «видит» пространство

def plot_decision_boundary(model, X, y):
    """Рисуем границы решений модели."""
    # Создаём сетку точек
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = torch.meshgrid(
        torch.linspace(x_min, x_max, 200),
        torch.linspace(y_min, y_max, 200),
        indexing='ij'
    )
    grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    
    # Предсказания для каждой точки сетки
    with torch.no_grad():
        Z = model(grid).argmax(dim=1).reshape(xx.shape)
    
    plt.figure(figsize=(7, 7))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
    for cls in range(3):
        mask = y == cls
        plt.scatter(X[mask, 0], X[mask, 1], s=10, label=f'Класс {cls}')
    plt.legend()
    plt.title('Границы решений модели')
    plt.axis('equal')
    plt.show()

plot_decision_boundary(model, X, y)

Круто, правда? Модель научилась разделять спирали! Но с одним скрытым слоем это может быть не идеально. Давайте усложним.

---
## Шаг 6. Чуть сложнее: расширяем пример

Добавим ещё один скрытый слой и посмотрим, как изменится качество. Также попробуем другую активацию и регуляризацию (Dropout).

In [ ]:
class DeeperNet(nn.Module):
    """Нейросеть с двумя скрытыми слоями и Dropout."""
    
    def __init__(self, input_size, hidden1, hidden2, output_size, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)  # dropout — регуляризация
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden2, output_size)
    
    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.fc3(x)
        return x


# Создаём «глубокую» модель
# попробуйте изменить: hidden1, hidden2, dropout
deeper_model = DeeperNet(
    input_size=2,
    hidden1=128,     # попробуйте 64, 256
    hidden2=64,      # попробуйте 32, 128
    output_size=3,
    dropout=0.1      # попробуйте 0.0, 0.2, 0.5
)

print(deeper_model)
print(f'Параметров: {sum(p.numel() for p in deeper_model.parameters()):,}')

**Что такое Dropout?** Это метод регуляризации: при обучении каждый нейрон «выключается» с вероятностью `p` (в нашем случае 0.1 = 10%). Это заставляет сеть не полагаться на конкретные нейроны и делает её более устойчивой. При оценке (инференсе) все нейроны работают, но их выходы умножаются на `(1 - p)`. Аналогия: если в команде каждый может заболеть, все учатся делать свою работу + чуть-чуть работу соседа.

In [ ]:
# Обучаем глубокую модель

torch.manual_seed(42)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(deeper_model.parameters(), lr=0.01)  # попробуйте lr=0.005 или 0.02

n_epochs = 800  # попробуйте изменить
losses_deep = []
accuracies_deep = []

for epoch in range(n_epochs):
    deeper_model.train()  # режим обучения (включает dropout)
    
    logits = deeper_model(X)
    loss = criterion(logits, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses_deep.append(loss.item())
    preds = logits.argmax(dim=1)
    acc = (preds == y).float().mean().item()
    accuracies_deep.append(acc)
    
    if (epoch + 1) % 200 == 0:
        print(f'Эпоха {epoch+1:4d} | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}')

# Оценка без dropout
deeper_model.eval()  # режим оценки (выключает dropout)
with torch.no_grad():
    logits = deeper_model(X)
    final_acc = (logits.argmax(dim=1) == y).float().mean().item()
print(f'\nИтого (eval): Accuracy={final_acc:.4f}')

In [ ]:
# Сравнение двух моделей

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(losses, label='SimpleNet (1 скрытый)', alpha=0.7)
axes[0].plot(losses_deep, label='DeeperNet (2 скрытых)', alpha=0.7)
axes[0].set_title('Loss: сравнение моделей')
axes[0].set_xlabel('Эпоха')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies, label='SimpleNet (1 скрытый)', alpha=0.7)
axes[1].plot(accuracies_deep, label='DeeperNet (2 скрытых)', alpha=0.7)
axes[1].set_title('Accuracy: сравнение моделей')
axes[1].set_xlabel('Эпоха')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Границы решений глубокой модели
plot_decision_boundary(deeper_model, X, y)

---
## Шаг 7. Практические задания / Эксперименты

Теперь ваша очередь! Вот 5 заданий — от простых к более сложным. Подсказки есть, но полных решений нет — это ваш ход.

### Задание 1. Исследуйте learning rate

Обучите `SimpleNet` с тремя разными значениями `learning_rate`: **0.1**, **0.01**, **0.001**. Сравните графики loss.

**Подсказка:** сохраняйте `losses` для каждого lr в отдельный список и постройте все три кривые на одном графике. Какой lr сходится быстрее? Какой даёт лучший результат?

### Задание 2. Поменяйте активацию

Замените `nn.ReLU()` в `SimpleNet` на `nn.Tanh()` или `nn.Sigmoid()`. Обучите и сравните с ReLU.

**Подсказка:** создайте новый класс или параметризуйте активацию через `__init__`. Обратите внимание, насколько быстрее/медленнее сходится обучение.

### Задание 3. Увеличьте количество классов

Измените `n_classes=5` в `generate_spiral()`. Как изменится качество? Сколько эпох теперь нужно?

**Подсказка:** больше классов = сложнее задача. Возможно, придётся увеличить `hidden_size` или количество эпох.

### Задание 4. Найдите ошибку в коде

Вот сломанный код обучения. Найдите и исправьте все ошибки:

```python
model = SimpleNet(2, 32, 3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()       # ошибка 1
    optimizer.step()      # ошибка 2
    # ошибка 3: чего не хватает для корректного логирования?
```

**Подсказка:** вспомните, что нужно делать с градиентами перед каждым шагом, и в каком порядке вызывать `zero_grad`, `backward`, `step`.

### Задание 5. Создайте свою архитектуру

Создайте модель с 3 скрытыми слоями и BatchNorm (`nn.BatchNorm1d`). BatchNorm нормализует активации внутри сети, что часто ускоряет и стабилизирует обучение.

**Подсказка:** `nn.BatchNorm1d(num_features)` ставится после линейного слоя и до активации: `Linear -> BatchNorm -> ReLU`. Сравните скорость сходимости с моделью без BatchNorm.

---
## Шаг 8. Идеи для дальнейшего изучения

Поздравляю! Вы освоили основы PyTorch. Вот куда двигаться дальше:

- **Свёрточные сети (CNN)** — для работы с изображениями. Начните с `nn.Conv2d` и датасета MNIST/CIFAR10.
- **DataLoader и Dataset** — для работы с большими данными и мини-батчами.
- **Transfer learning** — берём предобученную модель (ResNet, VGG) и дообучаем под свою задачу.
- **GPU-оптимизация** — смешанная точность (`torch.cuda.amp`), `torch.compile()` (PyTorch 2.0+).
- **Отладка обучения** — TensorBoard, lr schedulers, early stopping.

**Полезные ресурсы:**
- [Официальные туториалы PyTorch](https://pytorch.org/tutorials/)
- [PyTorch Examples на GitHub](https://github.com/pytorch/examples)
- Книга «Deep Learning with PyTorch» (Eli Stevens et al.)

Удачи в обучении!